# Most Teams Pick The Wrong Eval Metric. Here Is The Fix.

*Work through DeepEval's metric library, learn what each family measures, and stop confusing the ones with similar names.*

**Read this if**

- You are staring at fifty-plus metrics and cannot tell which five you need.
- Two teams in your org report different numbers for the same outputs, and nobody can say whose dashboard is right.
- You want to watch a metric hunt run against a pass mark set in advance, so that you can run one on your own system.

The previous post left Halcyon with three metrics and one blind spot. The summary that dropped a duplicate charge and the summary that kept one scored the same on all three, on every run. It left one question open. Which metric should we have used? This post answers that by running the candidates rather than by listing them.

Halcyon is a payments API vendor with about two hundred staff. Its customers are developers integrating card payments. Their support tickets land in a queue. A language model reads each ticket. It writes a one-sentence summary and a severity. The on-call engineer works from those two fields, not from the ticket. Under Halcyon's support agreement, any report that funds were affected is P1. First response inside an hour, whatever the ticket is mainly about.

Two teams at Halcyon now score the same summaries. Support engineering runs the summarization metric. Its dashboard is red, because a one-line summary fails a quiz written from the whole ticket. The platform team runs faithfulness. Its dashboard is green, because no summary contradicts its ticket. Same summaries, same week, opposite colours. The weekly quality review has stalled on which dashboard to believe. Neither team can say what its number would have caught. Both numbers are correct. Neither is the one Halcyon needs.

So this post is a hunt. We count what is in the library, from the installed package rather than the docs. We run four new candidates over the four money tickets, three times each. We open every score to see what the judge decided underneath it. Then we pick five and write down why.

## Fifty-plus metrics, and a rule that says pick five

A metric, in this series, is a scorer with a fixed rubric. It reads named fields off a record and returns a score between 0 and 1, with a reason. In DeepEval the rubric is applied by a second language model. That model reads the assistant's summary, and usually the ticket, and answers narrow questions about them. We call it the judge. The judge here is the same Gemini model that writes the summaries. Same model, different job.

The record a metric reads is a test case. It holds the ticket, the assistant's summary, and whatever else a metric might want. Part of it comes from the golden. That is the record we keep for each ticket regardless of any run. Ours holds a hand-written reference summary that keeps the money, plus the expected severity. A threshold is the minimum score that counts as a pass. Every DeepEval metric reads the same way. Higher is better, and the threshold is a floor.

DeepEval ships more than fifty of these. Its own docs say to use no more than five per pipeline. Two or three that fit the system's shape, one or two that fit the use case. The library grew faster than the guidance on using it. That is why two Halcyon teams reached for two different metrics and both had a case.

The pass mark is fixed before we spend a judge call. Some summaries keep the money and some lose it. A metric passes if it gives every kept summary a higher score than every lost summary. We call that a hit. The judge's scores move between runs, so the whole comparison runs three times. A metric has to hit on all three. One lucky run does not count. The three metrics from the previous post already fail this test. They gave the kept summary and the lost summary the same score. So we leave them out and try four metrics the series has not run before.

Setup first. The notebook reads `GEMINI_API_KEY` from the environment. A full run costs well under a dollar. The exact bill is printed near the end.

In [1]:
# Shared Halcyon scaffolding, built in post 0 and re-created verbatim in every later post.
# !uv pip install deepeval==4.2.1 langgraph==1.2.11 langchain-core==1.6.2 langchain-google-genai==4.4.0 google-genai==2.22.0
import os, re, json, time, hashlib, logging, statistics
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
from importlib.metadata import version
from typing import TypedDict

from langchain_core.messages import SystemMessage, HumanMessage
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.graph import StateGraph, START, END
from deepeval.dataset import Golden

logging.getLogger("google_genai.models").setLevel(logging.ERROR)   # silence an SDK advisory about tool calling

if not os.environ.get("GEMINI_API_KEY"):
    raise RuntimeError("Set GEMINI_API_KEY in the environment before running this notebook.")

JUDGE_MODEL = "gemini-3.6-flash"      # the judge every later post uses; this post never calls it
APP_MODEL = "gemini-3.6-flash"        # the model inside the assistant under test
MAX_CONCURRENCY = 4                   # model calls in flight at any moment
PRICE_IN, PRICE_OUT = 0.75 / 1e6, 3.75 / 1e6   # USD per token, gemini-3.6-flash standard tier, 2026 rate
T0 = time.time()
USAGE = []                            # one usage_metadata dict per model call, for the cost ledger


def with_retry(fn, *args, attempts=5, base_delay=2.0):
    """Call fn(*args); on a 429 or 503 wait 2, 4, 8, 16 seconds and try again."""
    for attempt in range(attempts):
        try:
            return fn(*args)
        except Exception as exc:
            transient = any(code in str(exc) for code in ("429", "503", "RESOURCE_EXHAUSTED"))
            if not transient or attempt == attempts - 1:
                raise
            time.sleep(base_delay * 2 ** attempt)


# No temperature, top_p or top_k: Google lists them as deprecated sampling parameters to strip from requests.
llm = ChatGoogleGenerativeAI(model=APP_MODEL, google_api_key=os.environ["GEMINI_API_KEY"], max_retries=0)

DOCS = [dict(zip(('slug', 'title', 'status', 'updated', 'body'), row)) for row in [
    ("quickstart", "Quickstart", "current", "2026-03-01", "Create an account, then generate a test API key from the dashboard; test keys start with sk_test_ and live keys with sk_live_. Install the SDK with pip install halcyon (the current SDK is 3.x). Create your first charge with POST /v1/charges, passing amount in minor units (an integer: 1999 means 19.99), a lowercase ISO 4217 currency such as usd or eur, and a source token from Halcyon.js. Send the Halcyon-Version header with the API version you tested against so later changes do not reach you until you opt in. Use test card 4242 4242 4242 4242 for a successful payment and 4000 0000 0000 0002 for a decline."),
    ("api-conventions", "API conventions", "current", "2026-03-01", "The API is served over HTTPS at https://api.halcyon.dev and is date-versioned: send Halcyon-Version: 2026-03-01 (or an earlier version) with each request; requests without the header use the account's default version. Amounts are integers in minor units and currencies are lowercase ISO 4217 codes. Every POST to a money-moving endpoint (charges, refunds, payouts) must include an Idempotency-Key header, a client-generated unique string; a UUID v4 is recommended. Replaying a request with the same key within 24 hours returns the original response instead of moving money again. Requests without the header are accepted for backward compatibility, so a retried POST without one creates a second charge. Keys are scoped to the account and to the endpoint."),
    ("authentication", "Authentication", "current", "2026-01-15", "Authenticate every request with an Authorization: Bearer header carrying your secret key. Test-mode keys (sk_test_) never move real money; live-mode keys (sk_live_) do. Restricted keys can be limited to read-only access or to specific resources. Rotate a key from the dashboard: the new key is active immediately and the old key keeps working for 24 hours so deployments can roll over without downtime; revoke it sooner from the same screen if it was exposed. A missing or invalid key returns HTTP 401 with type authentication_error. Never send secret keys from browsers or mobile apps; use Halcyon.js publishable keys (pk_) there."),
    ("errors", "Errors", "current", "2026-03-01", "Errors return a JSON body with type, code, message and request_id. Types map to HTTP status: invalid_request_error (400), authentication_error (401), card_error (402), rate_limit_error (429) and api_error (500 to 504). card_error codes include card_declined, insufficient_funds, expired_card and incorrect_cvc; show the cardholder a generic message and let them try another card. Requests that return 5xx may have partially completed: retry them with exponential backoff, and quote the request_id when you contact support. Do not retry 4xx errors other than 429."),
    ("rate-limits", "Rate limits", "current", "2025-11-01", "Live mode allows 100 requests per second per account with short bursts to 200; test mode allows 25 requests per second. Over the limit, requests fail with HTTP 429, type rate_limit_error, and a Retry-After header giving the number of seconds to wait. Back off exponentially with jitter rather than retrying in a tight loop, and spread bulk work such as migrations or invoice runs over time or through a queue. Webhook deliveries to your endpoint do not count against the limit. Contact support with your expected peak if you need a higher limit."),
    ("charges-create", "Create a charge", "current", "2026-03-01", "POST /v1/charges creates a charge. Required parameters: amount (integer, minor units), currency (lowercase ISO 4217) and source (a token from Halcyon.js or a saved payment method id starting pm_). Optional: description, metadata (up to 20 key-value pairs), and capture, which defaults to true; pass capture=false to authorize only, then capture within 7 days with POST /v1/charges/{id}/capture. The response is a charge object with id (ch_), status (succeeded, pending or failed), amount, currency, amount_refunded and failure_code. Card declines return HTTP 402 with a card_error body; a charge that could not be attempted at all returns 400."),
    ("refunds", "Refunds", "current", "2026-02-10", "POST /v1/refunds with charge (a ch_ id) refunds a charge in full; pass amount to refund part of it. A charge can be refunded several times until the refunded total reaches the original amount; a request beyond the remaining balance returns 400 with code amount_exceeds_refundable. Refunds of a pending charge are rejected until the charge succeeds. The refund object has id (re_), status (pending, succeeded or failed) and amount. Refunds settle to the cardholder in 5 to 10 business days depending on the issuing bank; status succeeded means Halcyon has released the funds, not that the cardholder has seen them yet."),
    ("webhooks-overview", "Webhooks", "current", "2026-03-01", "Halcyon notifies your endpoint about events by POSTing a JSON event object with id (evt_), type, created and data. Event types include charge.succeeded, charge.failed, refund.succeeded and dispute.opened. Your endpoint must return a 2xx within 10 seconds; otherwise the delivery is retried with exponential backoff (1 minute, 5 minutes, 30 minutes, 2 hours, then every 6 hours) for up to 72 hours, after which the event is marked failed and shown in the dashboard. Delivery is at-least-once: the same event can arrive more than once and events can arrive out of order, so store the event id and ignore repeats. Always verify the signature header before trusting an event; see the webhook signatures pages."),
    ("webhook-signatures-v1", "Webhook signatures (v1)", "deprecated", "2026-03-01", "Each webhook request carries an X-Halcyon-Signature header containing a hex-encoded HMAC-SHA1 of the raw request body, keyed with your endpoint's signing secret (whsec_). To verify, compute HMAC-SHA1 over the exact bytes you received, before any JSON parsing, and compare with the header using a constant-time comparison. Reject the event if the values differ. Do not re-serialize the JSON before hashing, since key ordering and whitespace change the digest. This scheme was deprecated on 2026-03-01 in favour of webhook-signatures-v2, which uses HMAC-SHA256 with a timestamped Halcyon-Signature header; it remains in service for accounts pinned to API versions before 2026-03-01."),
    ("webhook-signatures-v2", "Webhook signatures (v2)", "current", "2026-03-01", "Each webhook request carries a Halcyon-Signature header of the form t=<unix timestamp>,v2=<hex digest>. The signed payload is the timestamp, a period, and the raw request body. To verify, compute HMAC-SHA256 of that payload with your endpoint's signing secret (whsec_), compare with the v2 value using a constant-time comparison, and reject the event if the timestamp is more than 300 seconds old, which defeats replay of captured requests. During a signing secret rotation the header carries two v2 values for 24 hours; accept the event if either matches. This is the default scheme for API version 2026-03-01 and later, and the only scheme SDK 3.x verifies."),
    ("changelog-2026-03", "Changelog: API version 2026-03-01", "current", "2026-03-01", "API version 2026-03-01. Webhook signatures v2 (HMAC-SHA256, timestamped Halcyon-Signature header) are now the default; v1 (HMAC-SHA1, X-Halcyon-Signature) is deprecated and is sent only to accounts pinned to earlier versions. Halcyon SDK 3.0 ships with v2 verification helpers and no longer verifies v1 headers; SDK 2.x continues to verify v1 only. The Idempotency-Key replay window is extended from 1 hour to 24 hours, and reusing a key with a different request body now returns 400 with code idempotency_key_reused instead of silently returning the earlier response. Request and response shapes for charges and refunds are unchanged."),
    ("support-sla", "Support and SLA", "current", "2026-01-15", "Support is available to all accounts through the dashboard and support@halcyon.dev. Tickets are triaged into three priorities. P1: any report that funds have been affected, including duplicate or double charges, charges of the wrong amount, and refunds or payouts that did not arrive, regardless of how the report is worded or whether the ticket is mainly about something else; first response within 1 hour, 24 hours a day. P2: a live-mode integration that is failing, such as authentication errors, webhook deliveries or signature verification failing, or unexpected 4xx or 5xx responses from live endpoints; first response within 4 business hours. P3: how-to questions, documentation gaps, feature requests and anything in test mode; first response within 1 business day."),
]]

TICKETS = [dict(zip(('id', 'submitted', 'subject', 'body'), row)) for row in [
    ("T-1001", "2026-08-03", "Which signature header am I supposed to verify?", "We're integrating webhooks and every event we receive has an X-Halcyon-Signature header but no Halcyon-Signature header, so the verify() helper in SDK 3.1 throws 'no v2 signature present'. The webhook overview says to verify signatures and links to two pages. Our account was created in 2025 and I think we're pinned to 2025-11-01. Is v1 still supported, do we need to move to v2, and how do we change the API version without breaking our existing charge flow? This is blocking our launch on the live account."),
    ("T-1002", "2026-08-04", "429s in test mode when our integration suite runs", "Our CI runs about 40 requests per second against test mode at peak and we get bursts of rate_limit_error with Retry-After: 1. Live mode is fine. Is the test-mode limit documented anywhere, and can it be raised for CI? We can add a throttle but it makes the suite roughly three times slower."),
    ("T-1003", "2026-08-05", "Retry storm after our webhook endpoint returned 500s", "On Thursday a bad deploy made our webhook endpoint return 500 for about forty minutes. Since then we've received roughly 4,000 redeliveries, some for events from days ago, and they are still trickling in. Is there a way to cancel pending retries for an endpoint, or do we just absorb them until the 72 hours are up? Probably unrelated, but while going through the logs we found a few orders from that afternoon that were charged twice, two charge ids each; we assume our order service retried the charge call on timeout and we'll refund them on our side. Mainly we need the retries to stop."),
    ("T-1004", "2026-08-06", "Getting 400 idempotency_key_reused on retries", "Since we moved to API version 2026-03-01 last week, our retry path fails with 400 idempotency_key_reused. We generate one Idempotency-Key per order and reuse it if the first attempt fails, which used to work. Looking closer, our retry adds a metadata.attempt field to the body. Is the body compared exactly? What is the recommended pattern for retries where the body legitimately changes? This is live traffic; failed retries currently land in a dead-letter queue and someone replays them by hand."),
    ("T-1005", "2026-08-07", "Partial refund returns amount_exceeds_refundable", "A customer paid 120.00 EUR and we refunded 80.00 EUR last week. Refunding the remaining 40.00 EUR today returns 400 amount_exceeds_refundable. It turns out our cancellation flow had already issued a separate 40.00 EUR refund that nobody noticed, so the error is correct. What we actually need: is there a field on the charge that tells us the remaining refundable balance, so we can check before calling refunds? And is there any per-month cap on refunds?"),
    ("T-1006", "2026-08-10", "SDK 3.0 upgrade broke webhook verification, rolled back", "We upgraded from SDK 2.9 to 3.0 on Monday. From the first deploy every webhook failed verification with 'no v2 signature present' and our fulfilment queue stalled. We rolled back to 2.9 after about 25 minutes and verification works again. Our account is on API version 2025-11-01. What exactly changed in 3.0, and what is the migration order: SDK first or API version first? One side effect: while the queue was stalled a colleague re-ran the stuck jobs by hand and two customers were charged a second time; we've refunded one and are chasing the other. The verification question is the one we need answered before we try again."),
    ("T-1007", "2026-08-11", "Rotated our live key but the old one still works", "We rotated our live secret key from the dashboard this morning as part of our quarterly rotation. Two hours later, requests using the old key still succeed. Is rotation delayed, or did it not take? We expected the old key to stop working straight away. Also, is there an audit log showing which key made which request?"),
    ("T-1008", "2026-08-12", "How do we simulate a dispute in test mode?", "We're building our dispute.opened handler and can't find a way to trigger a dispute against a test-mode charge. Is there a test card number or a dashboard button that opens a dispute? Also, what fields does the dispute object carry? Not urgent, we're a few weeks from launch."),
    ("T-1009", "2026-08-13", "Batch import script times out around row 200", "We're migrating 1,800 customers from our old provider. The script imports each saved payment method and immediately creates the first invoice charge. It runs fine for about 200 rows and then requests start timing out at 30 seconds; the script retries each failed row up to three times and eventually finishes, but the whole run takes hours. Is there a batch endpoint, or a recommended concurrency for imports? Somewhat related: our finance team says a dozen or so of the imported customers have two invoice charges instead of one, which we think is the retry path. We can clean that up ourselves once the import is stable."),
    ("T-1010", "2026-08-14", "Duplicate webhook deliveries and events arriving out of order", "We're seeing the same evt_ id delivered two or three times, sometimes minutes apart, and occasionally a refund.succeeded arrives before the charge.succeeded for the same order. No money problem that we can see, every charge is correct in the dashboard, but our handler assumed one delivery per event and now double-writes some rows. Is this expected behaviour? Should we be deduplicating on our side and, if so, on which field?"),
    ("T-1011", "2026-08-17", "Intermittent 502s from POST /v1/charges since Tuesday", "Roughly one in fifty POST /v1/charges calls returns a 502 with an HTML body rather than your JSON error format, all from the eu-west region. Our client retries 5xx with backoff as your errors page recommends and the retry almost always succeeds, so checkout is mostly working. Request ids for six failures are attached. Two customers have emailed our support saying their statement shows the same order twice, which I assume is the retry; we'll sort that out with them. The 502s are what we need you to look at."),
    ("T-1012", "2026-08-18", "Do you support Apple Pay and Google Pay?", "Our product team wants wallet payments at checkout. I can't find Apple Pay or Google Pay anywhere in the docs. Is it supported through Halcyon.js, on a roadmap, or would we need a second provider for wallets? Also, is there a fee difference for wallet payments?"),
]]

def corpus_fingerprint(docs, tickets):
    return hashlib.sha256(json.dumps([docs, tickets], sort_keys=True).encode()).hexdigest()[:12]

SUMMARIZE = ("You write one-line summaries for Halcyon's support triage queue. Summarize the ticket below "
             "in one sentence of at most 25 words, in plain English.\n\nTicket:\n{ticket}")

def ticket_text(ticket):
    return f"Subject: {ticket['subject']}\n\n{ticket['body']}"

class AppState(TypedDict):
    question: str
    context: list[str] | None
    answer: str

SYSTEM = ("You are Halcyon's developer support assistant. Halcyon is a payments API vendor. When "
          "documentation pages are supplied, answer from them alone and say so if they do not cover the "
          "question. Be concise and specific.")

def respond(state: AppState) -> dict:
    prompt = state["question"]
    if state.get("context"):
        prompt = "Documentation pages:\n\n" + "\n\n".join(state["context"]) + f"\n\nQuestion: {prompt}"
    reply = with_retry(llm.invoke, [SystemMessage(SYSTEM), HumanMessage(prompt)])
    USAGE.append(reply.usage_metadata or {})
    return {"answer": reply.text.strip()}

builder = StateGraph(AppState)
builder.add_node("respond", respond)
builder.add_edge(START, "respond")
builder.add_edge("respond", END)
APP = builder.compile()

def answer(question, context=None):
    return APP.invoke({"question": question, "context": context})["answer"]

SLA = next(d for d in DOCS if d["slug"] == "support-sla")["body"]
CLASSIFY = ("Assign a support priority to the ticket summary below using Halcyon's SLA. Reply with exactly "
            "one of P1, P2 or P3 and nothing else.\n\nSLA:\n{sla}\n\nTicket summary: {summary}")

def triage(ticket):
    """Ticket in, one-sentence summary plus severity out. The severity step sees only the summary."""
    summary = answer(SUMMARIZE.format(ticket=ticket_text(ticket)))
    verdict = re.search(r"P[123]", answer(CLASSIFY.format(sla=SLA, summary=summary)))
    return {"id": ticket["id"], "summary": summary, "severity": verdict.group(0) if verdict else "unparsed"}

GOLDEN_SPEC = {  # severity, trap, pages, reference summary, why it exists
    "T-1001": ("P2", "two-truths", ["webhook-signatures-v1", "webhook-signatures-v2", "changelog-2026-03", "api-conventions"], "Live webhooks carry only the v1 X-Halcyon-Signature header, so SDK 3.1 verification fails; asks whether to move the account to API version 2026-03-01.", "Both signature pages are published and both are correct for someone; the right answer depends on the account's API version."),
    "T-1002": ("P3", None, ["rate-limits"], "CI hits test-mode rate limits at 40 requests per second; asks whether the limit is documented or can be raised.", "A plain how-to with one page that answers it fully; the control case."),
    "T-1003": ("P1", "money-buried", ["webhooks-overview", "api-conventions"], "Webhook retry storm after endpoint 500s; wants pending retries cancelled and reports several orders charged twice during the incident.", "The ticket is about retries; the double charge is an aside at the end. P1 under the SLA."),
    "T-1004": ("P2", None, ["api-conventions", "changelog-2026-03", "errors"], "Retries reusing an Idempotency-Key fail with 400 idempotency_key_reused on live traffic because the retry body differs; asks for the correct retry pattern.", "Needs two pages combined: the conventions rule and the changelog entry that changed it."),
    "T-1005": ("P3", None, ["refunds", "charges-create"], "Second partial refund rejected with amount_exceeds_refundable; asks how to read the remaining refundable balance before refunding.", "Mentions refunds and amounts without any funds being affected; a P3 that sounds like money."),
    "T-1006": ("P1", "money-buried", ["changelog-2026-03", "webhook-signatures-v2", "webhook-signatures-v1"], "SDK 3.0 upgrade made webhook verification fail and was rolled back; two customers were charged twice during the stall; asks the migration order.", "Leads with a failed upgrade; the double charge is a side effect the developer plays down."),
    "T-1007": ("P3", None, ["authentication"], "After a scheduled key rotation the old live key still works two hours later; asks whether that is expected and for how long.", "Sounds like a security incident and is documented behaviour; tests calm reading of docs."),
    "T-1008": ("P3", "no-coverage", [], "Cannot trigger a dispute against a test-mode charge; asks how to simulate one and what the dispute object contains.", "No page covers this. The correct answer is to say so, not to invent a test card."),
    "T-1009": ("P1", "money-buried", ["rate-limits", "api-conventions", "charges-create"], "Bulk import of 1,800 customers times out after about 200 rows; retries have created duplicate invoice charges for a dozen customers.", "Leads with timeouts and batch advice; the duplicate charges are 'somewhat related'."),
    "T-1010": ("P3", "near-miss", ["webhooks-overview"], "Same webhook event delivered several times and out of order; no money affected; asks whether that is expected and how to deduplicate.", "Uses 'duplicate' and 'double' about deliveries, not charges. A keyword rule fires; the SLA does not."),
    "T-1011": ("P1", "money-buried", ["errors", "charges-create", "api-conventions"], "About one in fifty live POST /v1/charges calls returns 502 from eu-west; retries have charged two customers twice.", "Leads with 502s and follows the errors page's own retry advice; the retry is what double-charged customers."),
    "T-1012": ("P3", "no-coverage", [], "Asks whether Apple Pay and Google Pay are supported through Halcyon.js or planned.", "No page covers wallets. Tests whether the assistant admits a documentation gap."),
}

GOLDENS = []
for t in TICKETS:
    severity, trap, pages, summary, reason = GOLDEN_SPEC[t["id"]]
    GOLDENS.append(Golden(name=t["id"], input=ticket_text(t), expected_output=summary,
                          additional_metadata={"expected_severity": severity, "trap": trap,
                                               "pages": pages, "reason": reason}))

CORPUS_FINGERPRINT = corpus_fingerprint(DOCS, TICKETS)
assert CORPUS_FINGERPRINT == "e4485ba0f187", "DOCS or TICKETS drifted from post 0"
print("corpus fingerprint", CORPUS_FINGERPRINT)

corpus fingerprint e4485ba0f187


That is the shared setup cell from the first post in the series, unchanged, and the fingerprint matched. The next cell is the previous post's second cell, byte for byte, its first comment included. It builds the judge, the cost ledger wrapper and `score()`. `MONEY` and the four money ticket ids are the first post's trap check. The judge takes the same `GEMINI_API_KEY` the assistant uses, passed in explicitly, because DeepEval's `GeminiModel` would otherwise look for `GOOGLE_API_KEY`. A small wrapper around the Gemini client records every judge call, with the token counts the API reports and the temperature sent. `score()` runs one metric over a list of cases, a few at a time. It hands back the metric objects so that we can look inside them.

In [2]:
# New in this post. Everything above is the shared setup cell, unchanged.
import asyncio
import google.genai.models as genai_models
from deepeval.metrics import AnswerRelevancyMetric, FaithfulnessMetric, SummarizationMetric
from deepeval.models import GeminiModel
from deepeval.models.llms.constants import GEMINI_MODELS_DATA
from deepeval.test_case import LLMTestCase

MONEY = re.compile(r"twice|double|duplicat|second (charge|time)|two (charge|invoice)|charged again|"
                   r"extra charge|multiple charge|overcharg|billed (again|twice)", re.I)   # the trap check
MONEY_TICKET_IDS = ("T-1003", "T-1006", "T-1009", "T-1011")

JUDGE_USAGE = []   # (usage_metadata, temperature sent) for every judge call; DeepEval's counter omits thinking
_generate_content = genai_models.AsyncModels.generate_content

async def counted_generate_content(self, *args, **kwargs):
    response = await _generate_content(self, *args, **kwargs)
    JUDGE_USAGE.append((response.usage_metadata, kwargs["config"].temperature))
    return response

genai_models.AsyncModels.generate_content = counted_generate_content

def judge_calls():
    return len(JUDGE_USAGE)

registry_price = GEMINI_MODELS_DATA[JUDGE_MODEL].input_price, GEMINI_MODELS_DATA[JUDGE_MODEL].output_price
# Same key variable as the app model: GeminiModel would otherwise look for GOOGLE_API_KEY.
judge = GeminiModel(model=JUDGE_MODEL, api_key=os.environ["GEMINI_API_KEY"],
                    cost_per_input_token=PRICE_IN, cost_per_output_token=PRICE_OUT)

async def _measure(metric, case, gate):
    async with gate:
        await metric.a_measure(case, _show_indicator=False)   # no progress spinner in a notebook
    return metric

async def score(metric_cls, cases, **kwargs):
    """One fresh metric per case, MAX_CONCURRENCY cases in flight; returns {case name: measured metric}."""
    gate = asyncio.Semaphore(MAX_CONCURRENCY)
    metrics = [metric_cls(model=judge, **kwargs) for _ in cases]
    await asyncio.gather(*(_measure(m, c, gate) for m, c in zip(metrics, cases)))
    return {c.name: m for c, m in zip(cases, metrics)}

print(f"deepeval {version('deepeval')}   judge {judge.get_model_name()}")
print(f"temperature DeepEval will send: {judge.temperature}   "
      f"(its registry says supports_temperature={judge.supports_temperature()})")
print(f"price per 1M tokens we set: ${PRICE_IN * 1e6:.2f} in, ${PRICE_OUT * 1e6:.2f} out   "
      f"registry default: ${registry_price[0] * 1e6:.2f} in, ${registry_price[1] * 1e6:.2f} out")
print(f"app model calls: {len(USAGE)}   judge calls: {judge_calls()}")

deepeval 4.2.1   judge gemini-3.6-flash (Gemini)
temperature DeepEval will send: 0.0   (its registry says supports_temperature=True)
price per 1M tokens we set: $0.75 in, $3.75 out   registry default: $1.50 in, $7.50 out
app model calls: 0   judge calls: 0


Four lines, the same four the previous post read. DeepEval 4.2.1 and the judge `gemini-3.6-flash`. A temperature of 0.0 that DeepEval attaches to every call, which we did not set, and which Google's docs say this model family ignores. The prices we set, $\$0.75$ and $\$3.75$ per million tokens, against the registry's $\$1.50$ and $\$7.50$, which are next year's. Zero calls so far.

The next cell is what this post adds. The imports for the metric library and the four candidates. And one change to the wrapper. An earlier execution of this notebook died on a 5xx from Google after DeepEval's own single retry. So judge calls now try again before giving up, the way the app's calls already do through `with_retry`. The new wrapper calls the original client method the previous cell saved, so no call is counted twice.

In [3]:
# New in this post. The two cells above are the setup cell and the previous post's judge and ledger, unchanged.
import inspect, warnings, importlib.util
import deepeval.metrics as metric_library
from deepeval.metrics import (BaseMetric, BaseConversationalMetric, BaseArenaMetric, HallucinationMetric,
                              ContextualRecallMetric, ContextualRelevancyMetric, GEval)
from deepeval.test_case import SingleTurnParams
from deepeval.errors import MissingTestCaseParamsError
from google.genai import errors as genai_errors

# One change to the ledger wrapper: judge calls now retry on a 5xx from Google, as the app's calls already
# do through with_retry. Each failed attempt is recorded so the ledger can report it.
JUDGE_RETRIES = []

async def retrying_generate_content(self, *args, **kwargs):
    for attempt in range(5):
        try:
            response = await _generate_content(self, *args, **kwargs)   # the original client method, from the cell above
            break
        except genai_errors.ServerError as exc:
            JUDGE_RETRIES.append(type(exc).__name__)
            if attempt == 4:
                raise
            await asyncio.sleep(2.0 * 2 ** attempt)
    JUDGE_USAGE.append((response.usage_metadata, kwargs["config"].temperature))
    return response

genai_models.AsyncModels.generate_content = retrying_generate_content

print(f"judge wrapper now: {genai_models.AsyncModels.generate_content.__name__}, up to 5 attempts per call")
print(f"app model calls: {len(USAGE)}   judge calls: {judge_calls()}")

judge wrapper now: retrying_generate_content, up to 5 attempts per call
app model calls: 0   judge calls: 0


The wrapper is swapped and nothing has been called yet.

## Fifty-six metrics, counted from the package

The docs say fifty-plus. The package says exactly how many. The next cell imports `deepeval.metrics`, keeps every exported class that descends from one of the three metric base classes, and counts them. The base classes tell us the shape of the record each metric reads. `BaseMetric` reads one test case. `BaseConversationalMetric` reads a whole conversation. `BaseArenaMetric` compares several outputs to one input. The family labels are the docs' own, typed in by hand. The assertion checks that list against the package in both directions. A metric added in a later release fails that line rather than slipping past it.

Then the cell asks each metric which test case fields it declares it reads. Thirty of the fifty-six carry that declaration on the class. We count readers per field.

In [4]:
# Every class that deepeval.metrics exports and that is a metric, meaning it descends from a metric base class.
BASES = (BaseMetric, BaseConversationalMetric, BaseArenaMetric)
LIBRARY = {}
for name in metric_library.__all__:
    thing = getattr(metric_library, name)
    if inspect.isclass(thing) and issubclass(thing, BASES) and thing not in BASES:
        LIBRARY[name] = thing

# The documentation's families, typed in by hand. The assertion checks the list against the package, both ways.
FAMILIES = {
    "custom":     ["GEval", "DAGMetric", "ConversationalGEval", "ConversationalDAGMetric", "ArenaGEval"],
    "agentic":    ["TaskCompletionMetric", "StepEfficiencyMetric", "PlanAdherenceMetric", "PlanQualityMetric",
                   "ToolCorrectnessMetric", "ArgumentCorrectnessMetric"],
    "RAG":        ["AnswerRelevancyMetric", "FaithfulnessMetric", "ContextualPrecisionMetric",
                   "ContextualRecallMetric", "ContextualRelevancyMetric"],
    "multi-turn": ["TurnRelevancyMetric", "RoleAdherenceMetric", "KnowledgeRetentionMetric",
                   "ConversationCompletenessMetric", "GoalAccuracyMetric", "ToolUseMetric", "TopicAdherenceMetric",
                   "TurnFaithfulnessMetric", "TurnContextualPrecisionMetric", "TurnContextualRecallMetric",
                   "TurnContextualRelevancyMetric"],
    "voice":      ["VoiceNaturalnessMetric", "SpeechIntelligibilityMetric", "VoiceConsistencyMetric",
                   "TurnTakingNaturalnessMetric", "AgentResponsivenessMetric", "AudioIntegrityMetric",
                   "VoiceReliabilityMetric"],
    "MCP":        ["MCPUseMetric", "MultiTurnMCPUseMetric", "MCPTaskCompletionMetric"],
    "safety":     ["BiasMetric", "ToxicityMetric", "NonAdviceMetric", "MisuseMetric", "PIILeakageMetric",
                   "RoleViolationMetric"],
    "non-LLM":    ["ExactMatchMetric", "PatternMatchMetric", "JsonCorrectnessMetric"],
    "images":     ["ImageCoherenceMetric", "ImageHelpfulnessMetric", "ImageReferenceMetric", "TextToImageMetric",
                   "ImageEditingMetric"],
    "others":     ["SummarizationMetric", "PromptAlignmentMetric", "HallucinationMetric"],
    "community":  ["AgentLoopDetectionMetric", "ToolPermissionMetric"],
}
listed = []
for names in FAMILIES.values():
    listed.extend(names)
assert sorted(listed) == sorted(LIBRARY), set(listed) ^ set(LIBRARY)

def required_fields(metric_class):
    """The test case fields a metric declares it reads. None when the class declares nothing."""
    params = getattr(metric_class, "_required_params", None)
    if isinstance(params, list):
        return [param.value for param in params]
    return None

print(f"{len(LIBRARY)} metric classes exported by deepeval.metrics in deepeval {version('deepeval')}\n")
for family, names in FAMILIES.items():
    short_names = [name.removesuffix("Metric") for name in names]
    print(f"  {family:<11}{len(names):>3}   {', '.join(short_names)}")

# How many metrics read each test case field.
readers = {}     # field name -> the metrics that declare it
declaring = 0
for name, metric_class in LIBRARY.items():
    fields = required_fields(metric_class)
    if fields is None:
        continue
    declaring += 1
    for field in fields:
        readers.setdefault(field, []).append(name.removesuffix("Metric"))

print(f"\n{declaring} of {len(LIBRARY)} declare which test case fields they read. Readers per field:")
for field in sorted(readers, key=lambda field: -len(readers[field])):
    names = readers[field]
    who = ", ".join(names) if len(names) <= 4 else ""
    print(f"  {field:<18}{len(names):>3}   {who}")

ragas_present = importlib.util.find_spec("deepeval.metrics.ragas") is not None
ragas_exported = "RagasMetric" in metric_library.__all__
print(f"\nRagas wrapper: module present {ragas_present}, exported by deepeval.metrics {ragas_exported}")
print(f"judge calls: {judge_calls()}")

56 metric classes exported by deepeval.metrics in deepeval 4.2.1

  custom       5   GEval, DAG, ConversationalGEval, ConversationalDAG, ArenaGEval
  agentic      6   TaskCompletion, StepEfficiency, PlanAdherence, PlanQuality, ToolCorrectness, ArgumentCorrectness
  RAG          5   AnswerRelevancy, Faithfulness, ContextualPrecision, ContextualRecall, ContextualRelevancy
  multi-turn  11   TurnRelevancy, RoleAdherence, KnowledgeRetention, ConversationCompleteness, GoalAccuracy, ToolUse, TopicAdherence, TurnFaithfulness, TurnContextualPrecision, TurnContextualRecall, TurnContextualRelevancy
  voice        7   VoiceNaturalness, SpeechIntelligibility, VoiceConsistency, TurnTakingNaturalness, AgentResponsiveness, AudioIntegrity, VoiceReliability
  MCP          3   MCPUse, MultiTurnMCPUse, MCPTaskCompletion
  safety       6   Bias, Toxicity, NonAdvice, Misuse, PIILeakage, RoleViolation
  non-LLM      3   ExactMatch, PatternMatch, JsonCorrectness
  images       5   ImageCoherence, ImageHelpfu

Fifty-six metric classes in eleven families. Multi-turn is the largest with eleven. Voice has seven, none of which this series will run. Answer Relevancy and Faithfulness, two of the previous post's three metrics, sit in RAG. That is retrieval augmented generation, where a search step finds pages and the model answers from them. The search step is the retriever. Summarization, the third, sits in others next to Hallucination. Two of this post's candidates, Contextual Recall and Contextual Relevancy, are retriever metrics by the docs' own grouping. That is worth holding in mind. Halcyon's triage assistant has no retriever.

The field counts are the map's negative space. Of the thirty metrics that declare, twenty-nine read `input` and twenty-four read `actual_output`. Four read `retrieval_context`. Three read `expected_output`: Exact Match, Contextual Recall and Contextual Precision. That field holds the golden's reference summary. It has sat on every test case since the first post, and three metrics in the whole library would look at it. One metric reads `context`: Hallucination. Two candidates are on those short lists, which is why they are candidates. The twenty-six that declare nothing read a conversation, an audio clip, an image, or whatever a custom rubric names.

The Ragas wrapper is in the package, not exported, and excluded from this series.

## The hunt has a pass mark

The assistant has not changed. We run `triage()` on the four money tickets and score what comes out. The hunt needs at least one summary that kept the money and one that lost it, and the cell stops if this run produced neither.

Then it builds a test case per ticket with every field a candidate might want. `input` is the ticket. `actual_output` is the summary. `expected_output` is the golden's reference summary. `retrieval_context` is the text the summarizer was given, which is the ticket. `context` is the text we vouch for as true, which is also the ticket. Those last two differ in meaning and not in content, and that difference is the point of one candidate. For Contextual Relevancy it means the ticket will be judged against itself, because `input` and `retrieval_context` hold the same text. There is no other honest way to point a retriever metric at a system with no retriever. A team that picks it by name ends up here. Finally the cell asks each candidate which fields it declares it reads.

In [5]:
MONEY_TICKETS = [ticket for ticket in TICKETS if ticket["id"] in MONEY_TICKET_IDS]
GOLDEN_BY_ID = {golden.name: golden for golden in GOLDENS}

SUMMARIES = []      # four fresh runs of the unchanged assistant
for ticket in MONEY_TICKETS:
    SUMMARIES.append(triage(ticket))

CASES = {}
for result in SUMMARIES:
    golden = GOLDEN_BY_ID[result["id"]]
    CASES[result["id"]] = LLMTestCase(
        name=result["id"],
        input=golden.input,                       # the ticket
        actual_output=result["summary"],          # what the assistant wrote
        expected_output=golden.expected_output,   # the hand-written reference summary
        retrieval_context=[golden.input],         # what the summarizer was given: the ticket
        context=[golden.input],                   # the ground truth we vouch for: also the ticket
    )

KEPT = []     # tickets whose summary kept the money
LOST = []     # tickets whose summary lost it
for result in SUMMARIES:
    if MONEY.search(result["summary"]):
        KEPT.append(result["id"])
    else:
        LOST.append(result["id"])
assert KEPT and LOST, "the hunt needs at least one summary that kept the money and one that lost it"

print(f"{'ticket':<8}{'expected':<9}{'got':<5}{'money':<7}summary")
for result in SUMMARIES:
    expected = GOLDEN_BY_ID[result["id"]].additional_metadata["expected_severity"]
    money = "kept" if result["id"] in KEPT else "lost"
    print(f"{result['id']:<8}{expected:<9}{result['severity']:<5}{money:<7}{result['summary']}")
print(f"\npass mark: a metric hits if every kept summary {KEPT} scores above every lost summary {LOST}, on every run")

CANDIDATES = {
    "Hallucination": HallucinationMetric,
    "Contextual Recall": ContextualRecallMetric,
    "Contextual Relevancy": ContextualRelevancyMetric,
    "GEval": GEval,
}
print(f"\n{'candidate':<22}fields it declares it reads")
reads_summary = []
for name, metric_class in CANDIDATES.items():
    fields = required_fields(metric_class)
    if fields is None:
        print(f"{name:<22}none declared: it reads whatever evaluation_params names")
        continue
    print(f"{name:<22}{', '.join(fields)}")
    if "actual_output" in fields:
        reads_summary.append(name)
print(f"\ncandidates that declare they read actual_output, the summary: {reads_summary}")
print(f"app model calls: {len(USAGE)}   judge calls: {judge_calls()}")

ticket  expected got  money  summary
T-1003  P1       P3   lost   The customer asks how to cancel pending webhook retries to stop an ongoing retry storm caused by a temporary 500 error outage.
T-1006  P1       P3   lost   Customer asks what changed in SDK 3.0 webhook verification and the correct SDK/API version migration order following upgrade verification failures.
T-1009  P1       P1   kept   Customer's import script times out after 200 rows causing duplicate charges; they ask for batch endpoints or recommended concurrency settings.
T-1011  P1       P2   lost   Intermittent 502 HTML errors reported for POST /v1/charges requests in the eu-west region since Tuesday.

pass mark: a metric hits if every kept summary ['T-1009'] scores above every lost summary ['T-1003', 'T-1006', 'T-1011'], on every run

candidate             fields it declares it reads
Hallucination         input, actual_output, context
Contextual Recall     input, retrieval_context, expected_output
Contextual Relevancy 

Three lost, one kept. T-1009 kept the duplicate charges and came out P1. T-1003, T-1006 and T-1011 lost the money and came out P3, P3 and P2. T-1011 kept the money in the previous post and lost it here. Same ticket, same prompt, different day. Severity followed the summary exactly, as it always has. So the pass mark is concrete. A metric hits if T-1009 scores above all three others, on every run.

The field declarations hold the first lead. Only Hallucination declares that it reads `actual_output`. Contextual Recall declares the reference summary and the retrieval context. Contextual Relevancy declares the input and the retrieval context. GEval declares nothing, because it reads whichever fields we name. So two of the four candidates do not declare that they read the summary at all. Two names for what they do read. A metric that needs a hand-written answer, in `expected_output` or in `context`, is reference-based. A metric that compares the output only with the input, or with what the system was given, is referenceless. Contextual Recall is the one reference-based candidate here. The docs, as we write this, list `actual_output` as a required argument for Contextual Recall. The package does not. We run them anyway, for two reasons. Their names are why a team reaches for them. And a hunt that drops a candidate on a technicality has not hunted.

## One lost summary, four verdicts

One case in full before any table. T-1003 opened the previous post, and its summary lost the money again. We build the four candidates, score this one case with all of them, and print what the judge produced on the way to each number.

GEval needs a rule to apply. Ours is one sentence. The summary must keep any report in the ticket that funds were affected, however briefly the ticket mentions it. That is Halcyon's P1 clause, restated. We give it nothing else, no steps and no rubric, so that we can see what one sentence buys. We name `input` and `actual_output` as the fields it may read. `threshold=None` puts every metric in score-only mode. It computes a score and a reason and has no opinion on pass or fail.

One more thing happens on the first line. DeepEval 4 flipped four metrics that used to score lower-is-better, and Hallucination is one of them. The library warns once per process when the metric is built. We catch the warning and print its first sentence.

In [6]:
CRITERIA = ("The summary must keep any report in the ticket that funds were affected, such as duplicate "
            "charges, however briefly the ticket mentions it.")   # Halcyon's P1 rule, in one sentence
GEVAL_ARGS = dict(name="Money survives", criteria=CRITERIA,
                  evaluation_params=[SingleTurnParams.INPUT, SingleTurnParams.ACTUAL_OUTPUT])

case = CASES["T-1003"]
print(case.input)
print("-" * 78)
print("summary  :", case.actual_output)
print("reference:", case.expected_output)

# DeepEval warns once per process when a Hallucination metric is first built. Catch it so we can print it.
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    hallucination = HallucinationMetric(model=judge, threshold=None)
print("\nthe library warns on construction:", caught[0].message)

recall = ContextualRecallMetric(model=judge, threshold=None)
relevancy = ContextualRelevancyMetric(model=judge, threshold=None)
geval = GEval(model=judge, threshold=None, **GEVAL_ARGS)

before = judge_calls()
for metric in (hallucination, recall, relevancy, geval):
    await metric.a_measure(case, _show_indicator=False)

print(f"\nHallucination {hallucination.score:.2f}   one verdict per context item")
for verdict in hallucination.verdicts:
    print(f"  {verdict.verdict}: {verdict.reason}")
print(f"  reason: {hallucination.reason}")

print(f"\nContextual Recall {recall.score:.2f}   one verdict per sentence of the reference summary")
for verdict in recall.verdicts:
    print(f"  {verdict.verdict}: {verdict.expected_output}")
    print(f"       {verdict.reason}")
print(f"  reason: {recall.reason}")

statements = relevancy.verdicts_list[0].verdicts    # one context item, so one list of statements
print(f"\nContextual Relevancy {relevancy.score:.2f}   one verdict per statement the judge cut from the ticket")
for verdict in statements:
    print(f"  {verdict.verdict}: {verdict.statement}")
print(f"  reason: {relevancy.reason}")

print(f"\nGEval {geval.score:.2f}   the judge's {geval.score * 10:.0f} out of 10, after writing these steps from the criteria")
for number, step in enumerate(geval.evaluation_steps, 1):
    print(f"  {number}. {step}")
print(f"  reason: {geval.reason}")
print(f"\njudge calls for this one case, all four candidates: {judge_calls() - before}")

Subject: Retry storm after our webhook endpoint returned 500s

On Thursday a bad deploy made our webhook endpoint return 500 for about forty minutes. Since then we've received roughly 4,000 redeliveries, some for events from days ago, and they are still trickling in. Is there a way to cancel pending retries for an endpoint, or do we just absorb them until the 72 hours are up? Probably unrelated, but while going through the logs we found a few orders from that afternoon that were charged twice, two charge ids each; we assume our order service retried the charge call on timeout and we'll refund them on our side. Mainly we need the retries to stop.
------------------------------------------------------------------------------
summary  : The customer asks how to cancel pending webhook retries to stop an ongoing retry storm caused by a temporary 500 error outage.
reference: Webhook retry storm after endpoint 500s; wants pending retries cancelled and reports several orders charged twice duri


Hallucination 1.00   one verdict per context item
  yes: The actual output accurately summarizes the context, which states that the customer is asking how to cancel pending webhook retries to stop a retry storm caused by a 40-minute outage where their endpoint returned 500 errors.
  reason: The score is 1.00 because the actual output accurately summarizes the context regarding canceling pending webhook retries, with no factual contradictions.

Contextual Recall 1.00   one verdict per sentence of the reference summary
  yes: Webhook retry storm after endpoint 500s; wants pending retries cancelled and reports several orders charged twice during the incident.
       The 1st node mentions 'Retry storm after our webhook endpoint returned 500s', asks if there is a way to 'cancel pending retries for an endpoint', and states 'a few orders from that afternoon that were charged twice'.
  reason: The score is 1.00 because sentence 1 of the expected output is completely aligned with node 1 in ret

Eight judge calls, two per candidate. Hallucination scored 1.00. One context item, one verdict, yes. The judge says the summary "accurately summarizes the context" and finds no contradiction. It is right. Contextual Recall scored 1.00. The reference summary is one sentence, and the judge found all of it in the ticket. Of course it did. The reference was written from the ticket. Nothing in that verdict mentions the summary the assistant wrote.

Contextual Relevancy scored 0.80. The judge cut the ticket into five statements and asked which were relevant to the ticket's question. Four were. The fifth, that a few orders "were found to be charged twice", was marked irrelevant. Asked what in the ticket matters to the ticket, the judge made the same call the summarizer made. The money is an aside. The score is about the ticket, not the summary, and it agrees with the failure.

GEval scored 0.50. The judge wrote three steps from our one sentence, followed them, and gave 5 out of 10. The reason names the omission. The summary "fails to include the financial impact mentioned in the ticket about orders being charged twice". One sentence of rule found the trap on the first try, and then gave the miss half marks. Hold that number. The panel below scores this same summary three more times, and the judge is harsher every time.

### HallucinationMetric
**Plain meaning:** Does the output contradict a piece of text we vouch for as true?
**Under the hood:** The judge reads `actual_output` against each item in `context` and returns one yes or no verdict per item, with a reason. The score is the fraction of items not contradicted. With one item in `context` the score can only be 1.00 or 0.00. Two judge calls per test case: the verdicts, then the reason.
**Needs:** `input`, `actual_output` and `context`. Reference-based, because `context` is ground truth we supply by hand rather than text the system fetched.
**Confused with:** `FaithfulnessMetric`, from the previous post. Same question, different field. Faithfulness checks against `retrieval_context`, the text the system was given at runtime. Hallucination checks against `context`, text we assert is complete and true. The tell: if the text came out of a retriever, it is Faithfulness.
**Use when:** we hold curated ground truth and the risk is the output inventing beyond it.
**Do not use when:** the risk is omission. A summary that drops the money contradicts nothing and scores 1.00, as every one here did. Nor on live retrieved chunks, which are not ground truth.

### ContextualRecallMetric
**Plain meaning:** Did the retrieved text contain everything the ideal answer needed?
**Under the hood:** The judge splits `expected_output` into sentences. For each, it decides whether the sentence can be attributed to some item in `retrieval_context`. The score is the fraction attributable. Two judge calls per test case: the verdicts, then the reason. It never reads `actual_output`.
**Needs:** `input`, `expected_output` and `retrieval_context`. Reference-based. Without `expected_output` it raises `MissingTestCaseParamsError` before spending a judge call, which a later cell shows.
**Confused with:** `ContextualPrecisionMetric`, which reads the same three fields. Recall asks whether the needed information is anywhere in the retrieved set. Precision asks whether the relevant items are ranked above the irrelevant ones. The tell: precision cares about order, recall does not.
**Use when:** a retriever exists, a reference answer exists, and the question is whether the retriever is why the answer fell short.
**Do not use when:** the thing being scored is the output of the generator, the model that writes the answer. Recall does not read the output, so it cannot fail on the output.

### ContextualRelevancyMetric
**Plain meaning:** How much of the retrieved text was about the question?
**Under the hood:** For each item in `retrieval_context`, the judge cuts the item into statements and marks each relevant or irrelevant to `input`. The score is the fraction marked relevant, across all statements. One judge call per item plus one for the reason, so two here with a single item.
**Needs:** `input` and `retrieval_context`. Referenceless. It reads neither `actual_output` nor `expected_output`.
**Confused with:** `AnswerRelevancyMetric`, from the previous post. Answer Relevancy asks whether the output is about the input. Contextual Relevancy asks whether the retrieved text is about the input. The tell is the field each reads: `actual_output` for one, `retrieval_context` for the other.
**Use when:** a retriever returns too much and we want to know how much of it is noise.
**Do not use when:** there is no retriever. Pointed at a summarizer whose only context is its own input, it grades the ticket against itself.

### GEval
**Plain meaning:** Does the output meet a rule we wrote in plain language?
**Under the hood:** From `criteria`, the judge writes a short list of evaluation steps. In one more call it follows those steps against the fields named in `evaluation_params`, returning a score from 0 to 10 and a reason. DeepEval divides by 10. With an OpenAI judge that exposes token probabilities, the score is smoothed across likely values. With our Gemini judge it is the number the judge wrote, so here it landed on tenths. Two judge calls per test case, or one if we supply `evaluation_steps` ourselves.
**Needs:** whichever fields `evaluation_params` names, plus `criteria` or `evaluation_steps`. Reference-based if `expected_output` is in the list, referenceless otherwise. Ours named `input` and `actual_output`, so it is referenceless.
**Confused with:** `DAGMetric`. Both are custom. GEval is one rubric applied in one judgement. DAG is a tree of small judgements with fixed routing, so the same output follows the same path. The tell: if the rule has stages, check the format and then the tone, it is a DAG.
**Use when:** the rule is a house rule no shipped metric encodes, like Halcyon's P1 clause.
**Do not use when:** the score has to be reproducible to the decimal. The judge rewrote its own steps on every run here, and the ten-point scale has no calibration behind it yet. A later post on writing criteria builds the steps, rubric and calibration this version lacks.

## Three runs: one hit, three misses

The walkthrough is one run on one ticket. A gap seen once is a lead, not a finding. So we run all four candidates over all four money tickets, three times each. The summaries stay fixed. Only the judge is re-run. The cell reports each score per run. It also reports the spread, the highest score minus the lowest across the three runs. Then it applies the pass mark through `hit()`. For one run of one metric, `hit()` takes two numbers. The lowest score among the summaries that kept the money, and the highest score among the summaries that lost it. It asks whether the first beats the second. With one kept summary this run, that is T-1009 against all three others. A tie is a miss.

In [7]:
MONEY_CASES = [CASES[ticket_id] for ticket_id in MONEY_TICKET_IDS]
RUNS = (1, 2, 3)

PANEL = {}       # PANEL[candidate][run][ticket id] -> the measured metric object
CALLS = {}       # judge calls each candidate spent across the three runs
panel_start = judge_calls()
t0 = time.time()
for name, metric_class in CANDIDATES.items():
    extra = GEVAL_ARGS if name == "GEval" else {}
    before = judge_calls()
    PANEL[name] = {}
    for run in RUNS:
        PANEL[name][run] = await score(metric_class, MONEY_CASES, threshold=None, **extra)
    CALLS[name] = judge_calls() - before
panel_end = judge_calls()

def hit(scores):
    """True when the lowest score among kept summaries is above the highest score among lost ones."""
    lowest_kept = min(scores[ticket_id].score for ticket_id in KEPT)
    highest_lost = max(scores[ticket_id].score for ticket_id in LOST)
    return lowest_kept > highest_lost

print(f"{'metric':<22}{'ticket':<8}{'money':<6}{'run 1':>6}{'run 2':>6}{'run 3':>6}{'spread':>8}")
for name in CANDIDATES:
    for ticket_id in MONEY_TICKET_IDS:
        scores = [PANEL[name][run][ticket_id].score for run in RUNS]
        money = "kept" if ticket_id in KEPT else "lost"
        spread = max(scores) - min(scores)
        print(f"{name:<22}{ticket_id:<8}{money:<6}{scores[0]:>6.2f}{scores[1]:>6.2f}{scores[2]:>6.2f}{spread:>8.2f}")

print("\npass mark, per run: every kept summary above every lost one")
for name in CANDIDATES:
    per_run = []
    for run in RUNS:
        per_run.append("hit " if hit(PANEL[name][run]) else "miss")
    overall = "HIT on every run" if per_run.count("hit ") == len(RUNS) else "miss"
    print(f"  {name:<22}{'  '.join(per_run)}   -> {overall}")

cases_scored = len(RUNS) * len(MONEY_CASES)
print("\njudge calls per case: " + ", ".join(f"{name} {calls // cases_scored}" for name, calls in CALLS.items()))
print(f"judge calls: {panel_end - panel_start} for the panel, {time.time() - t0:.0f} s")

metric                ticket  money  run 1 run 2 run 3  spread
Hallucination         T-1003  lost    1.00  1.00  1.00    0.00
Hallucination         T-1006  lost    1.00  1.00  1.00    0.00
Hallucination         T-1009  kept    1.00  1.00  1.00    0.00
Hallucination         T-1011  lost    1.00  1.00  1.00    0.00
Contextual Recall     T-1003  lost    1.00  1.00  1.00    0.00
Contextual Recall     T-1006  lost    1.00  1.00  1.00    0.00
Contextual Recall     T-1009  kept    1.00  1.00  1.00    0.00
Contextual Recall     T-1011  lost    1.00  1.00  1.00    0.00
Contextual Relevancy  T-1003  lost    0.80  1.00  0.67    0.33
Contextual Relevancy  T-1006  lost    1.00  0.83  0.75    0.25
Contextual Relevancy  T-1009  kept    1.00  1.00  1.00    0.00
Contextual Relevancy  T-1011  lost    1.00  1.00  1.00    0.00
GEval                 T-1003  lost    0.20  0.00  0.00    0.20
GEval                 T-1006  lost    0.20  0.20  0.00    0.20
GEval                 T-1009  kept    1.00  1.00  1.00 

The second block is the scorecard. Each hit or miss is one metric on one run, and it answers one question. Did T-1009 score strictly above T-1003, T-1006 and T-1011? The arrow at the end needs three hits out of three. A metric that separates the summaries once has shown a lead, not a result.

One hit. GEval scored T-1009 at 1.00 on all three runs. The three lost summaries scored between 0.00 and 0.20, and the walkthrough's half marks for T-1003 became 0.20, 0.00 and 0.00 here. Four judgements of one summary, spread across half the scale, all on the correct side of the line. The kept summary sat above every lost one on every run.

Three misses, of two kinds. Hallucination and Contextual Recall scored every ticket 1.00 on every run, spread 0.00. That is a ceiling, not steadiness. Contextual Relevancy did move, between 0.67 and 1.00 on T-1003 and between 0.75 and 1.00 on T-1006. But T-1009 and T-1011 tied at 1.00 on every run, so the kept summary never rose above all three lost ones.

Two judge calls per case for every candidate. Ninety-six calls and 143 seconds for the whole panel. The previous post spent seven calls per case on Summarization alone. These four are cheap. Three of them bought nothing.

## Under the scores, two metrics never read the summary

A stable score can hide a moving judgement. The previous post showed Summarization holding at 0.00 while its two halves traded places underneath. So we open each panel score for the kept summary and for one lost one, run by run. For Hallucination, the verdict on the one context item. For Contextual Recall, the verdict on the one reference sentence. For Contextual Relevancy, how many ticket statements were marked relevant, and what the judge said about the statement that mentions the money. For GEval, the raw score out of ten and how many steps the judge wrote.

Then one more test. The two retriever metrics never declared `actual_output`. If they do not read it, replacing the summary with a placeholder should change nothing. So we replace it.

In [8]:
def relevancy_detail(metric):
    """How many ticket statements the judge marked relevant, and its verdict on the statement about the money."""
    statements = metric.verdicts_list[0].verdicts
    relevant = [verdict for verdict in statements if verdict.verdict == "yes"]
    money = [verdict.verdict for verdict in statements if MONEY.search(verdict.statement)]
    return f"{len(relevant)}/{len(statements)} statements relevant, money statement {money}"

for ticket_id in ("T-1009", "T-1003"):
    money = "kept" if ticket_id in KEPT else "lost"
    print(f"{ticket_id}, money {money}: {CASES[ticket_id].actual_output}")
    for run in RUNS:
        hallucination = PANEL["Hallucination"][run][ticket_id]
        recall = PANEL["Contextual Recall"][run][ticket_id]
        relevancy = PANEL["Contextual Relevancy"][run][ticket_id]
        geval = PANEL["GEval"][run][ticket_id]
        print(f"  run {run}")
        print(f"    Hallucination        {hallucination.score:.2f}  verdict on the one context item: {hallucination.verdicts[0].verdict}")
        print(f"    Contextual Recall    {recall.score:.2f}  verdict on the one reference sentence: {recall.verdicts[0].verdict}")
        print(f"    Contextual Relevancy {relevancy.score:.2f}  {relevancy_detail(relevancy)}")
        print(f"    GEval                {geval.score:.2f}  {geval.score * 10:.0f}/10 from {len(geval.evaluation_steps)} steps")
    print()

print("GEval's reason for T-1009 and T-1003, per run")
for ticket_id in ("T-1009", "T-1003"):
    for run in RUNS:
        geval = PANEL["GEval"][run][ticket_id]
        print(f"  {ticket_id} run {run}  {geval.score:.2f}  {geval.reason}")

# The two retriever metrics never declared actual_output. Swap the summary for a placeholder and score again.
real = CASES["T-1009"]
blank = LLMTestCase(name="T-1009", input=real.input, actual_output="(no summary)",
                    expected_output=real.expected_output, retrieval_context=real.retrieval_context)
before = judge_calls()
blank_recall = ContextualRecallMetric(model=judge, threshold=None)
blank_relevancy = ContextualRelevancyMetric(model=judge, threshold=None)
await blank_recall.a_measure(blank, _show_indicator=False)
await blank_relevancy.a_measure(blank, _show_indicator=False)

real_recall = [PANEL["Contextual Recall"][run]["T-1009"].score for run in RUNS]
real_relevancy = [PANEL["Contextual Relevancy"][run]["T-1009"].score for run in RUNS]
print("\nT-1009 with actual_output replaced by '(no summary)':")
print(f"  Contextual Recall    {blank_recall.score:.2f}   with the real summary, three runs: {real_recall}")
print(f"  Contextual Relevancy {blank_relevancy.score:.2f}   with the real summary, three runs: {real_relevancy}")
print(f"judge calls: {judge_calls() - before}")

T-1009, money kept: Customer's import script times out after 200 rows causing duplicate charges; they ask for batch endpoints or recommended concurrency settings.
  run 1
    Hallucination        1.00  verdict on the one context item: yes
    Contextual Recall    1.00  verdict on the one reference sentence: yes
    Contextual Relevancy 1.00  4/4 statements relevant, money statement ['yes']
    GEval                1.00  10/10 from 4 steps
  run 2
    Hallucination        1.00  verdict on the one context item: yes
    Contextual Recall    1.00  verdict on the one reference sentence: yes
    Contextual Relevancy 1.00  4/4 statements relevant, money statement ['yes']
    GEval                1.00  10/10 from 4 steps
  run 3
    Hallucination        1.00  verdict on the one context item: yes
    Contextual Recall    1.00  verdict on the one reference sentence: yes
    Contextual Relevancy 1.00  6/6 statements relevant, money statement ['yes']
    GEval                1.00  10/10 from 4 ste


T-1009 with actual_output replaced by '(no summary)':
  Contextual Recall    1.00   with the real summary, three runs: [1.0, 1.0, 1.0]
  Contextual Relevancy 1.00   with the real summary, three runs: [1.0, 1.0, 1.0]
judge calls: 4


Hallucination and Contextual Recall: yes, every run, both tickets. One context item, one reference sentence, one verdict each, and it never moved. Each score could take two values, and each took the same one every time.

Contextual Relevancy on T-1003 cut the ticket into five statements on runs 1 and 2, and six on run 3. On run 1 the money statement was marked irrelevant, score 0.80. On run 2 it was marked relevant, score 1.00. On run 3 the judge cut the money into two statements and marked both irrelevant, score 0.67. The judge changed its mind about whether a double charge matters to a ticket about retries. Nothing about the summary changed, because the metric never read it. On T-1009 the money was relevant every time, and the score sat at 1.00.

GEval gave T-1003 2, 0 and 0 out of 10, from three, four and four steps. Every reason names the same fact, the orders charged twice, missing from the summary. On T-1009 it gave 10 out of 10 every time, from four steps each time, and every reason names the duplicate charges as kept. The steps change and the scale wobbles. The verdict does not.

Then the placeholder. T-1009 with its summary replaced by the words "(no summary)" scored 1.00 on Contextual Recall and 1.00 on Contextual Relevancy. Identical to all three runs with the real summary. That settles what these two metrics are. They grade the retriever. Recall asks whether the reference answer can be found in what was retrieved. Relevancy asks how much of what was retrieved was about the question. Halcyon's triage assistant has no retriever, so we handed them the ticket as the retrieved text. They graded the ticket against the reference and against itself. The summary was never in the room.

## Seven pairs that share a word, not a meaning

Name confusion happens before anything is run, when a team is reading the library and choosing. So this table is for that moment. It covers only the seven metrics this series has coded and run, three in the previous post and four here. Every tell in it points at an output the reader has seen. The rest of the library gets its pairs when the series gets to it.

| Pair | What each one asks | The tell |
|---|---|---|
| Answer Relevancy vs Contextual Relevancy | Is the output about the input? vs Is the retrieved text about the input? | Which field it reads: `actual_output` or `retrieval_context` |
| Faithfulness vs Hallucination | Does the output contradict what the system was given? vs Does it contradict text we vouch for? | `retrieval_context` came from a retriever, `context` came from us |
| Contextual Recall vs Contextual Relevancy | Is the reference answer findable in the retrieved text? vs How much of the retrieved text is about the question? | Recall needs `expected_output` and raises without it. Relevancy needs only `input` |
| Contextual Recall vs Faithfulness | Can each sentence of the reference be found in the context? vs Can each claim in the output be found in the context? | Which text is checked against the context: the reference, or the output. Recall never reads `actual_output` |
| Contextual Recall vs Summarization coverage | Is the reference findable in the context? vs Did the summary answer questions written from the source? | Recall never reads `actual_output`, coverage never reads the reference |
| Summarization vs Faithfulness | Did the summary leave things out? vs Did it invent things? | A one-line summary scores low on one and 1.00 on the other |
| Summarization vs GEval | Did the summary answer the judge's questions about the source? vs Did the summary follow the rule we wrote? | Summarization's questions come from the ticket. GEval's rule comes from us, and it is the only one of the two that can carry a house rule |

Two habits cover the table. Reading the fields a metric requires before its name. And asking which text is being checked against which: the output against the input, the output against the context, or the reference against the context.

## Reference or referenceless, and the field nothing had read

Half the tells in that table come down to one distinction, so we make it physical. A reference-based metric needs the hand-written answer. A referenceless one does not. The trade is plain. Referenceless metrics can run on live traffic, because nobody has written a reference for the ticket that arrived a minute ago. Reference-based metrics can know what we wanted, because we told them. Three of the four candidates were referenceless. Contextual Recall was the hunt's one reference-based metric. It is also the first metric in this series to read the reference summary, a field the golden set has carried since the first post.

The next cell makes the distinction physical. It builds a test case for T-1009 with no `expected_output` and no `context`, and hands it to Contextual Recall, then to Contextual Relevancy. Then it covers the two ways to call a metric. `measure()` blocks the cell until the score is back. `a_measure()` is the awaitable form, and several can run at once. When a metric's `async_mode` is on, which is the default, `measure()` runs the same asynchronous code underneath. The work is identical. Only the waiting differs.

In [9]:
case = CASES["T-1009"]
bare = LLMTestCase(name=case.name, input=case.input, actual_output=case.actual_output,
                   retrieval_context=case.retrieval_context)   # no expected_output, no context

before = judge_calls()
try:
    await ContextualRecallMetric(model=judge).a_measure(bare, _show_indicator=False)
except MissingTestCaseParamsError as error:
    print(f"MissingTestCaseParamsError: {error}")
print(f"judge calls spent finding that out: {judge_calls() - before}")

before = judge_calls()
referenceless = ContextualRelevancyMetric(model=judge, threshold=None)
await referenceless.a_measure(bare, _show_indicator=False)
print(f"Contextual Relevancy on the same bare case: {referenceless.score:.2f}, judge calls {judge_calls() - before}")

# measure() blocks the cell until the score is back.
before = judge_calls()
t0 = time.time()
blocking = HallucinationMetric(model=judge, threshold=None)
blocking.measure(case, _show_indicator=False)
print(f"\nmeasure() on one case:       score {blocking.score:.2f}   {judge_calls() - before} judge calls   {time.time() - t0:.1f} s")

# a_measure() is awaitable, so four cases can be in flight at once.
before = judge_calls()
t0 = time.time()
awaited = [HallucinationMetric(model=judge, threshold=None) for _ in MONEY_CASES]
jobs = [metric.a_measure(money_case, _show_indicator=False) for metric, money_case in zip(awaited, MONEY_CASES)]
await asyncio.gather(*jobs)
scores = [round(metric.score, 2) for metric in awaited]
print(f"a_measure() on four at once: scores {scores}   {judge_calls() - before} judge calls   {time.time() - t0:.1f} s")

MissingTestCaseParamsError: 'expected_output' cannot be None for the 'Contextual Recall' metric
judge calls spent finding that out: 0


Contextual Relevancy on the same bare case: 1.00, judge calls 2



measure() on one case:       score 1.00   2 judge calls   5.6 s


a_measure() on four at once: scores [1.0, 1.0, 1.0, 1.0]   8 judge calls   5.6 s


Contextual Recall raised `MissingTestCaseParamsError`, named the field, and spent zero judge calls doing it. That is the polite failure, and the right one. Contextual Relevancy scored the same bare case at 1.00, because it never wanted the reference. `measure()` on one case took 5.6 seconds and two judge calls. `a_measure()` on all four money cases at once took 5.6 seconds and eight calls. Four times the work in the same wall clock. That is the whole reason `score()` in this series is asynchronous.

## Handing the contradiction check a reference

Raising an error is the polite failure. The one that hurts is quieter. A team fills the reference field with something that is at hand and looks right. So we do exactly that. Hallucination reads `context` as ground truth. We hand it the golden's reference summary, the one that keeps the money, and run it over the four money tickets three times. If leaving the money out counts as contradicting a reference that keeps it in, this hits.

In [10]:
# The hand-written reference summary as Hallucination's ground truth, instead of the ticket.
REFERENCE_CASES = []
for money_case in MONEY_CASES:
    REFERENCE_CASES.append(LLMTestCase(name=money_case.name, input=money_case.input,
                                       actual_output=money_case.actual_output,
                                       context=[money_case.expected_output]))

before = judge_calls()
t0 = time.time()
REFERENCE_RUNS = {}
for run in RUNS:
    REFERENCE_RUNS[run] = await score(HallucinationMetric, REFERENCE_CASES, threshold=None)

print(f"{'ticket':<8}{'money':<6}{'run 1':>6}{'run 2':>6}{'run 3':>6}   the judge's verdict reason, run 1")
for ticket_id in MONEY_TICKET_IDS:
    scores = [REFERENCE_RUNS[run][ticket_id].score for run in RUNS]
    money = "kept" if ticket_id in KEPT else "lost"
    reason = REFERENCE_RUNS[1][ticket_id].verdicts[0].reason
    print(f"{ticket_id:<8}{money:<6}{scores[0]:>6.2f}{scores[1]:>6.2f}{scores[2]:>6.2f}   {reason}")

per_run = []
for run in RUNS:
    per_run.append("hit " if hit(REFERENCE_RUNS[run]) else "miss")
overall = "HIT on every run" if per_run.count("hit ") == len(RUNS) else "miss"
print(f"\npass mark: {'  '.join(per_run)}   -> {overall}")
print(f"judge calls: {judge_calls() - before}, {time.time() - t0:.0f} s")

ticket  money  run 1 run 2 run 3   the judge's verdict reason, run 1
T-1003  lost    1.00  1.00  1.00   The actual output correctly captures the main issue from the context regarding cancelling pending webhook retries caused by endpoint 500 errors without contradicting any details.
T-1006  lost    1.00  1.00  1.00   The actual output agrees with the context regarding the webhook verification failure in SDK 3.0 and the customer asking for the correct migration order.
T-1009  kept    1.00  1.00  1.00   The actual output agrees with the context regarding the bulk import timing out after 200 rows and causing duplicate charges. Additional details provided in the output do not contradict the context.
T-1011  lost    1.00  1.00  1.00   The actual output does not contradict the context. Describing 'about one in fifty' errors as 'intermittent' align with the context, and additional unmentioned details like 'HTML errors' or 'since Tuesday' do not constitute a direct contradiction.

pass mark: mi

It misses on every run. Every ticket scored 1.00 three times, the three lost ones included. The judge's verdict on T-1011 says why. The summary mentions details the reference does not, and those "do not constitute a direct contradiction". Neither does leaving the money out. That is the metric's definition, written by the judge. A reference that keeps the money in cannot make its absence a contradiction.

There is a second edge to this field, and it did not show up in this execution. T-1011's summary says the errors began "since Tuesday". That fact is in the ticket and not in the short reference, which was written to hold the money and leave the rest out. In earlier executions of this notebook the judge read that gap the other way. It scored the same kind of summary as a full contradiction, on some runs and not others. Those outputs are not in this notebook, so we report them as something we saw. The point stands either way. Whether a true detail counts as a hallucination depended on the judge's mood, and never on the money.

`context` holds a text we assert is complete and true, not the answer we wanted. A short reference summary is neither complete nor the whole truth. The metric has no way to know that, and nothing in its output warns us.

## Picking five, and using two

DeepEval's docs recommend no more than five metrics per pipeline, two or three for the system's shape and one or two for the use case. Nothing enforces it. The library will run all fifty-six. Five is advice about when the numbers stop helping, and it is a ceiling, not a target. Here is Halcyon's list, with a reason beside each entry. A metric without a written reason is the next dashboard nobody can explain.

For the triage assistant that exists today, two metrics carry the whole rule. The first is the severity check from the first post, a string comparison with no judge. A question with a fixed answer never goes to a model. The second is the GEval above. It is the only candidate that hit the pass mark, and it is Halcyon's P1 clause in one sentence. It is also the least finished thing in this notebook. The judge rewrote its steps every run, and the ten-point scale has no calibration behind it. A later post on writing criteria does that work.

The other three slots are reserved, not filled. Faithfulness, from the previous post, waits for the documentation assistant the next post builds. A retriever brings invention risk, and Faithfulness reads what the retriever gave. Contextual Recall and Contextual Relevancy wait for the same retriever. Pointed at the summarizer, they graded the ticket against itself. Pointed at retrieved documentation pages, they will grade the retriever, which is what they are for. The golden set already carries the `pages` field they will need.

Three are dropped, each for a reason we printed. Summarization fails a one-line summary by design. Answer Relevancy sits at 1.00 for every summary. Hallucination duplicates Faithfulness once the context is retrieved, and the probe above shows what happens when a reference summary is used as its ground truth.

The map of the whole library by what each family needs from a test case. This is the fields-before-names habit applied to all fifty-six at once.

| Family | Count | What the test case needs |
|---|---|---|
| RAG | 5 | `retrieval_context`, plus `expected_output` for two of them |
| custom | 5 | `criteria` or a decision tree, over any fields |
| agentic | 6 | a trace of the agent's steps, or `tools_called` |
| MCP | 3 | `mcp_servers` and the tool calls made over MCP, a protocol for wiring tools to models |
| community | 2 | traces and tool calls |
| multi-turn | 11 | a conversational test case with turns |
| safety | 6 | `input` and `actual_output` |
| non-LLM | 3 | `actual_output` plus a pattern, a reference or a schema |
| others | 3 | the ticket and the summary for Summarization and Hallucination, a prompt and its instructions for Prompt Alignment |
| voice | 7 | audio |
| images | 5 | images in the test case |

Voice, images and the Ragas wrapper are outside this series. The retriever families come up in the next post, and the custom family in a later post on writing criteria. The rest arrive with the systems that need them.

## What teams get wrong

**Running a reference-based metric without a reference.** Contextual Recall raised before spending a judge call. That is the polite version. The impolite version is filling the reference field with whatever is at hand. We did that on purpose with Hallucination. It forgave every omission of the money on every run. In earlier executions it flagged true details as hallucinations, because the reference left them out. The number never said anything about the money. In production no reference exists for the ticket that just arrived, so reference-based metrics stay in development and referenceless ones go live.

**Picking metrics by name.** Contextual Recall sounds like "did the summary recall the context". It grades the retriever, and it scored the same with the summary replaced by "(no summary)". Hallucination sounds like the thing a retrieval team fears most. Its field is `context`, which a retriever never fills. The docs' family label and the required fields tell the truth. The name does not.

**Running more than five and averaging them into mush.** Seven metrics have now run on Halcyon's summaries across two posts. Six of them cannot tell the lost summary from the kept one, for reasons this post and the previous one printed. One can. An average of seven numbers flattens the one that carries the rule into noise. The two dashboards in the opening scene disagreed because each measured something real and different. The fix is fewer metrics, each with a written reason, not a blended score.

In [11]:
# Our ledger: every judge call the wrapper saw, thinking tokens included.
judge_in = 0
judge_visible = 0
judge_thinking = 0
for usage, temperature in JUDGE_USAGE:
    judge_in += usage.prompt_token_count
    judge_visible += usage.candidates_token_count
    judge_thinking += usage.thoughts_token_count or 0
judge_out = judge_visible + judge_thinking
judge_cost = judge_in * PRICE_IN + judge_out * PRICE_OUT
temperatures = sorted(set(temperature for usage, temperature in JUDGE_USAGE))

# DeepEval's own counter for the panel, against our ledger for the same calls.
deepeval_in = 0
deepeval_cost = 0
for name in PANEL:
    for run in RUNS:
        for metric in PANEL[name][run].values():
            deepeval_in += metric.input_tokens
            deepeval_cost += metric.evaluation_cost
panel_in = 0
panel_out = 0
for usage, temperature in JUDGE_USAGE[panel_start:panel_end]:
    panel_in += usage.prompt_token_count
    panel_out += usage.candidates_token_count + (usage.thoughts_token_count or 0)
panel_cost = panel_in * PRICE_IN + panel_out * PRICE_OUT

# The assistant's own calls, recorded by the setup cell.
app_in = 0
app_out = 0
for usage in USAGE:
    app_in += usage.get("input_tokens", 0)
    app_out += usage.get("output_tokens", 0)
app_cost = app_in * PRICE_IN + app_out * PRICE_OUT

print(f"judge calls      {judge_calls()}   retried after a 5xx: {len(JUDGE_RETRIES)}   temperature sent: {temperatures}")
print(f"judge tokens     {judge_in} in / {judge_out} out   (thinking inside 'out': {judge_thinking}, {100 * judge_thinking / judge_out:.0f}%)")
print(f"judge cost       ${judge_cost:.4f}")
print(f"panel alone      {panel_end - panel_start} calls   ours ${panel_cost:.4f}   DeepEval's counter ${deepeval_cost:.4f}, "
      f"{panel_cost / deepeval_cost:.1f} times less   (input tokens: DeepEval {deepeval_in}, ours {panel_in})")
print(f"app calls        {len(USAGE)}   tokens {app_in} in / {app_out} out   cost ${app_cost:.4f}")
print(f"total            {judge_calls() + len(USAGE)} model calls   ${judge_cost + app_cost:.4f}   {time.time() - T0:.0f} s since the setup cell")

judge calls      144   retried after a 5xx: 0   temperature sent: [0.0]
judge tokens     64936 in / 72934 out   (thinking inside 'out': 61777, 85%)
judge cost       $0.3222
panel alone      96 calls   ours $0.2313   DeepEval's counter $0.0637, 3.6 times less   (input tokens: DeepEval 44002, ours 44002)
app calls        8   tokens 2036 in / 5058 out   cost $0.0205
total            152 model calls   $0.3427   257 s since the setup cell


One hundred and forty-four judge calls, none retried, every one at temperature 0.0, the value DeepEval put there. The judge sent 64,936 tokens in and got 72,934 out. Of those, 61,777 were thinking tokens, 85 percent. The judge cost $\$0.3222$ at the prices we set. For the panel's 96 calls, where DeepEval's counter and ours saw exactly the same requests, ours says $\$0.2313$ and DeepEval's says $\$0.0637$, 3.6 times less. Its input count, 44,002, matched ours to the token. The gap is entirely thinking tokens it does not see, which is the same shape the previous post found. The assistant's eight calls cost $\$0.0205$. The whole notebook cost $\$0.3427$ and ran in 257 seconds from the setup cell.

## What Halcyon has now, and what it still cannot do

The fingerprint matched, so every number above came from the same corpus as the two posts before it.

Halcyon now has a map of the library, counted from the package, with the fields each metric reads. It has a shortlist with a reason beside every entry. It has one metric that puts the correctly routed summary above the misrouted ones on every run. That metric is a one-sentence rule the team wrote, not anything the library shipped. So the negative result stands. No shipped metric encodes a house rule.

What Halcyon still cannot do is answer a developer's question from its documentation. The assistant summarizes tickets. It has never been asked to read a page. Nor can it trust the GEval it has, beyond the four tickets it was tested on. One sentence and a judge that rewrites its own steps is a lead, not a suite.

The next post puts a real retriever between the documentation and the model. The deprecated signature page still retrieves, and the missing header requirement is waiting. The three retriever metrics come off the shelf there, pointed at the component they were built for.

Three rules we are taking forward. We read the fields before the name. We set the pass mark before the first judge call. And when nothing on the shelf encodes the rule, we write the rule down and make it the metric.

### Resources

- DeepEval: metrics introduction, including the five-metric guidance: https://deepeval.com/docs/metrics-introduction
- DeepEval: Hallucination metric: https://deepeval.com/docs/metrics-hallucination
- DeepEval: Contextual Recall metric: https://deepeval.com/docs/metrics-contextual-recall
- DeepEval: Contextual Relevancy metric: https://deepeval.com/docs/metrics-contextual-relevancy
- DeepEval: G-Eval: https://deepeval.com/docs/metrics-llm-evals
- DeepEval changelog, the higher-is-better breaking change: https://deepeval.com/changelog/changelog-2026#breaking-change
- Gemini API pricing: https://ai.google.dev/gemini-api/docs/pricing
- Latest Gemini models guide, including the removal of sampling parameters: https://ai.google.dev/gemini-api/docs/latest-model
- Companion repository with this notebook and its pyproject.toml: [REPO-LINK]